# Acquire and audit three public coffee datasets
Downloads exact Roboflow versions for Capstone v1, Lulus v1, and Niacubilla v1; stores immutable TAR archives and SHA256 manifests on Drive; then runs the public-dataset eligibility audit. **CPU only. No training or checkpoint evaluation.**


In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')
import hashlib, importlib, json, os, shutil, subprocess, sys, tarfile, time
from pathlib import Path
REPO=Path('/content/coffee-bean-detection'); BRANCH='codex/public-dataset-eligibility-audit'
REMOTE='https://'+'github.com/ediprin/coffee-bean-detection.git'
os.chdir('/content')
if REPO.exists(): shutil.rmtree(REPO)
clone_error=''
for attempt in range(1,4):
    clone=subprocess.run(['git','clone','--depth','1','--branch',BRANCH,REMOTE,str(REPO)],text=True,capture_output=True)
    if clone.returncode==0: break
    clone_error=(clone.stderr or clone.stdout).strip(); print(f'CLONE ATTEMPT {attempt}/3 GAGAL:',clone_error)
    if REPO.exists(): shutil.rmtree(REPO)
else: raise RuntimeError('GitHub tidak dapat diakses setelah 3 percobaan: '+clone_error)
subprocess.run([sys.executable,'-m','pip','install','-q','-e',str(REPO),'roboflow'],check=True)
sys.path.insert(0,str(REPO/'src')); importlib.invalidate_caches(); os.chdir(REPO)
from coffee_detector.analysis.public_dataset_eligibility import audit_public_dataset_registry, extract_audit_archive
from coffee_detector.drive_project import resolve_drive_project_root
PROJECT_ROOT=resolve_drive_project_root(required_relative_paths=('bundles/coffee-detection-with-standard-v8-yolov8.tar',))
print('REPO:',subprocess.check_output(['git','rev-parse','--short','HEAD'],text=True).strip())
print('PROJECT:',PROJECT_ROOT)


In [ ]:
SOURCES=[
 {'code':'capstone_v1','workspace':'capstone-2-wwe5t','project':'coffee-bean-defect-a0vno','version':1},
 {'code':'lulus_v1','workspace':'lulus-vpibo','project':'green-coffee-bean-defects','version':1},
 {'code':'niacubilla_v1','workspace':'niacubilla','project':'coffee-bean-defects','version':1},
]
BUNDLES=PROJECT_ROOT/'bundles'; EVIDENCE=PROJECT_ROOT/'evidence/public-dataset-v2-audit'
BUNDLES.mkdir(parents=True,exist_ok=True); (EVIDENCE/'acquisition').mkdir(parents=True,exist_ok=True)
LOCAL_BASE=Path('/content/public-dataset-v2-downloads'); LOCAL_BASE.mkdir(exist_ok=True)
def file_sha256(path):
    digest=hashlib.sha256()
    with path.open('rb') as stream:
        for block in iter(lambda:stream.read(8*1024*1024),b''): digest.update(block)
    return digest.hexdigest()
api_key=userdata.get('ROBOFLOW_API_KEY')
if not api_key: raise RuntimeError('Tambahkan Colab Secret ROBOFLOW_API_KEY dan aktifkan akses notebook.')
from roboflow import Roboflow
rf=Roboflow(api_key=api_key)
print('ACQUISITION CONTRACT:',[(s['code'],s['version']) for s in SOURCES])


In [ ]:
ROOTS={}; ARCHIVES={}; acquisition=[]
for source in SOURCES:
    code=source['code']; local=LOCAL_BASE/code
    bundle=BUNDLES/f"{source['workspace']}--{source['project']}--v{source['version']}--yolov8.tar"
    manifest_path=EVIDENCE/'acquisition'/f'{code}.json'
    if bundle.is_file() and manifest_path.is_file():
        manifest=json.loads(manifest_path.read_text(encoding='utf-8'))
        actual=file_sha256(bundle)
        if actual!=manifest.get('archive_sha256'): raise RuntimeError(f'{code}: SHA bundle Drive berbeda dari manifest')
        root=extract_audit_archive(bundle,LOCAL_BASE/f'{code}-restored')
        print('RESTORE VERIFIED:',code,actual)
    else:
        if bundle.exists() or manifest_path.exists(): raise RuntimeError(f'{code}: bundle/manifest parsial; jangan timpa, periksa Drive')
        if local.exists(): shutil.rmtree(local)
        print('DOWNLOAD EXACT VERSION:',code,source['workspace'],source['project'],source['version'])
        downloaded=rf.workspace(source['workspace']).project(source['project']).version(source['version']).download('yolov8',location=str(local),overwrite=True)
        root=Path(downloaded.location).resolve()
        yaml_files=sorted(root.rglob('data.yaml'))
        if len(yaml_files)!=1: raise RuntimeError(f'{code}: data.yaml harus tepat satu, ditemukan {yaml_files}')
        with tarfile.open(bundle,'w') as archive: archive.add(root,arcname=code)
        actual=file_sha256(bundle)
        manifest={**source,'format':'yolov8','downloaded_at_unix':int(time.time()),'archive':str(bundle.relative_to(PROJECT_ROOT)),'archive_bytes':bundle.stat().st_size,'archive_sha256':actual,'training_executed':False}
        manifest_path.write_text(json.dumps(manifest,indent=2,ensure_ascii=False),encoding='utf-8')
        print('SAVED VERIFIED:',code,actual)
    ROOTS[code]=root; ARCHIVES[code]=bundle; acquisition.append(manifest)
print(json.dumps(acquisition,indent=2,ensure_ascii=False))


In [ ]:
import yaml
registry=yaml.safe_load((REPO/'configs/public_dataset_audit/v2_candidate_registry.yaml').read_text(encoding='utf-8'))
by_code={row['code']:row for row in registry['datasets']}
for manifest in acquisition:
    row=by_code[manifest['code']]
    row['dataset_root']=str(ROOTS[manifest['code']]); row['archive_path']=str(ARCHIVES[manifest['code']]); row['archive_sha256']=manifest['archive_sha256']
standard=BUNDLES/'coffee-detection-with-standard-v8-yolov8.tar'
if standard.is_file():
    ROOTS['coffee_standard_v8']=extract_audit_archive(standard,LOCAL_BASE/'coffee_standard_v8')
    ARCHIVES['coffee_standard_v8']=standard
    by_code['coffee_standard_v8']['dataset_root']=str(ROOTS['coffee_standard_v8']); by_code['coffee_standard_v8']['archive_path']=str(standard)
FROZEN_REGISTRY=EVIDENCE/'v2_acquired_registry.yaml'
FROZEN_REGISTRY.write_text(yaml.safe_dump(registry,sort_keys=False,allow_unicode=True),encoding='utf-8')
result=audit_public_dataset_registry(FROZEN_REGISTRY,EVIDENCE/'reports',root_overrides=ROOTS,archive_overrides=ARCHIVES,near_threshold=4)
print('DECISION:',result['decision'])
print('AUDITED:',result['audited_dataset_count'],'/',result['dataset_count'])
print('ELIGIBLE LINEAGES:',result['eligible_lineage_count'])
print('ELIGIBLE NEAR-DUP CANDIDATES:',result['eligible_near_cross_dataset_candidates'])
for row in result['datasets']: print(row['code'],row['status'],row.get('images',0),row.get('boxes',0),row.get('reasons',[]))
print('TRAINING AUTHORIZED:',result['training_authorized'])
print('SUMMARY:',EVIDENCE/'reports/public_dataset_eligibility_summary.json')


In [ ]:
import pandas as pd
from IPython.display import display
display(pd.DataFrame([{k:row.get(k) for k in ('code','status','images','boxes','class_count','estimated_source_parents','exact_cross_split_groups','parent_cross_split_groups','near_cross_split_candidates','reasons')} for row in result['datasets']]))
display(pd.DataFrame(result['cross_dataset']['near_candidates_by_dataset_pair']))
print('Kirim dua tabel dan decision. Jangan training.')
